In [3]:
# GitHub collaboration network
# Assignment notebook for Social Graphs 2025
# Author: Oliver Badike Hansen


In [3]:


import os
import aiohttp
import asyncio
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib import colormaps as cmaps
from matplotlib.lines import Line2D
import matplotlib.colors as mcolors
from pyvis.network import Network
import logging
import sys
import nest_asyncio
from dotenv import load_dotenv


nest_asyncio.apply()



In [5]:
# -----------------------------------------------------
# Logging Configuration
# -----------------------------------------------------
# Create a custom formatter that shows depth, user, and repo information
class GitHubGraphFormatter(logging.Formatter):
    def format(self, record):
        # Extract custom attributes if they exist
        depth = getattr(record, 'depth', None)
        user = getattr(record, 'user', None)
        repo = getattr(record, 'repo', None)
        time = self.formatTime(record, self.datefmt)

        # Create prefix based on available information
        prefix = ""
        if depth is not None:
            prefix += f"[DEPTH:{depth}]"
        if user is not None:
            prefix += f"[USER:{user}]"
        if repo is not None:
            prefix += f"[REPO:{repo}]"

        if prefix:
            prefix += " "

        # Format the message with the prefix
        return f"[{time}] {prefix}{super().format(record)}"

# Configure logger
logger = logging.getLogger("github_graph")
logger.setLevel(logging.INFO)  # Set to DEBUG to see more detailed logs

# Prevent duplicate log messages
logger.propagate = False
if logger.handlers:
    logger.handlers.clear()

# Create console handler for INFO and above
console_handler = logging.StreamHandler(sys.stdout)
console_handler.setLevel(logging.INFO)

# Create formatter and add it to the handler
formatter = GitHubGraphFormatter('%(levelname)s: %(message)s')
console_handler.setFormatter(formatter)

# Add handler to logger
logger.addHandler(console_handler)

# Optionally, add a file handler to capture all logs including DEBUG
file_handler = logging.FileHandler('github_graph.log', mode='w')
file_handler.setLevel(logging.DEBUG)
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)

logger.info("Logging system initialized")
logger.debug("Debug logging enabled - check github_graph.log for detailed logs")

# Function to control log verbosity
def set_log_level(level):
    """
    Set the logging level for both console and file handlers.

    Parameters:
    -----------
    level : str or int
        The logging level to set. Can be a string ('DEBUG', 'INFO', 'WARNING', 'ERROR', 'CRITICAL')
        or the corresponding integer value.
    """
    if isinstance(level, str):
        level = getattr(logging, level.upper())

    logger.setLevel(level)
    console_handler.setLevel(level)

    level_name = logging.getLevelName(level)
    logger.info(f"Console log level set to {level_name}")

    # Example usage:
    # set_log_level('DEBUG')  # Show all logs including debug messages
    # set_log_level('INFO')   # Show info, warning, error (default)
    # set_log_level('WARNING')  # Show only warnings and errors




[2025-10-29 20:41:56,239] INFO: Logging system initialized


In [6]:
async def get_repos(session, user, limit=20):
    """Fetch repositories owned by a user."""
    url = f"{BASE}/users/{user}/repos"
    logger.debug(f"Fetching repositories for user '{user}' (limit: {limit})", 
                extra={"user": user})

    async with session.get(url, headers=HEADERS, params={"per_page": limit}) as response:
        if response.status == 200:
            data = await response.json()
            repos = [repo["name"] for repo in data]
            logger.debug(f"Successfully fetched {len(repos)} repositories for user '{user}'", 
                        extra={"user": user})
            return repos
        elif response.status == 403 and 'X-RateLimit-Remaining' in response.headers and int(response.headers['X-RateLimit-Remaining']) == 0:
            logger.error(f"Rate limit exceeded for user '{user}'", extra={"user": user})
            raise RateLimitExceeded(f"GitHub API rate limit exceeded for user '{user}'")
        else:
            logger.warning(f"Failed to fetch repositories for user '{user}': HTTP {response.status}", 
                          extra={"user": user})
            response.raise_for_status()
            return []

async def get_contributors(session, user, repo, limit=20):
    """Fetch contributors for a repository."""
    url = f"{BASE}/repos/{user}/{repo}/contributors"
    logger.debug(f"Fetching contributors for repository '{user}/{repo}' (limit: {limit})", 
                extra={"user": user, "repo": repo})

    async with session.get(url, headers=HEADERS, params={"per_page": limit}) as response:
        if response.status == 200:
            data = await response.json()
            contributors = [c["login"] for c in data]
            logger.debug(f"Successfully fetched {len(contributors)} contributors for '{user}/{repo}'", 
                        extra={"user": user, "repo": repo})
            return contributors
        elif response.status == 404:
            logger.warning(f"Repository '{user}/{repo}' not found (HTTP 404)", 
                          extra={"user": user, "repo": repo})
            return []
        elif response.status == 403 and 'X-RateLimit-Remaining' in response.headers and int(response.headers['X-RateLimit-Remaining']) == 0:
            logger.error(f"Rate limit exceeded for repository '{user}/{repo}'", extra={"user": user, "repo": repo})
            raise RateLimitExceeded(f"GitHub API rate limit exceeded for repository '{user}/{repo}'")
        else:
            logger.warning(f"Failed to fetch contributors for '{user}/{repo}': HTTP {response.status}", 
                          extra={"user": user, "repo": repo})
            response.raise_for_status()
            return []

# Custom exception for rate limit
class RateLimitExceeded(Exception):
    """Exception raised when GitHub API rate limit is exceeded."""
    pass


In [7]:
def save_graph(G, graph_file):
    """Save the graph to a file, converting set attributes to strings for GEXF compatibility."""
    if not graph_file:
        graph_file = GRAPH_FILE

    logger.info(f"Saving current progress to {graph_file}")
    G_save = G.copy()

    for node, data in G_save.nodes(data=True):
        for key, val in list(data.items()):
            if isinstance(val, set):
                data[key] = ", ".join(sorted(val))

    for node, data in G_save.nodes(data=True):
        G_save.nodes[node]["label"] = node
        if isinstance(data.get("repos"), set):
            G_save.nodes[node]["repos"] = ", ".join(sorted(data["repos"]))
        elif "repos" not in data:
            G_save.nodes[node]["repos"] = ""

    nx.write_gexf(G_save, graph_file)
    logger.info(f"Saved graph with {len(G_save.nodes())} nodes and {len(G_save.edges())} edges to {graph_file}")
    return G_save

def save_exploration_state(G, visited, exploration_queue, state_file=None):
    """
    Save the current exploration state to resume later.

    Parameters:
    -----------
    G : nx.Graph
        The current graph
    visited : set
        Set of visited users
    exploration_queue : list
        List of (user, depth) tuples representing users to explore
    state_file : str, optional
        Path to save the state file (default: "exploration_state.pkl")
    """
    import pickle

    if not state_file:
        state_file = "exploration_state.pkl"

    # Save the graph first
    graph_file = state_file.replace(".pkl", ".gexf")
    save_graph(G, graph_file)

    # Save the exploration state
    state = {
        "visited": visited,
        "queue": exploration_queue,
        "graph_file": graph_file
    }

    with open(state_file, "wb") as f:
        pickle.dump(state, f)

    logger.info(f"Saved exploration state with {len(visited)} visited users and {len(exploration_queue)} users in queue to {state_file}")
    return state


In [8]:
import aiohttp
import networkx as nx

async def build_graph(seed_user, depth=2, repo_limit=10, contrib_limit=40, graph_file=None, state_file=None):
    """
    Explore users and their collaborators with support for resuming after rate limits.
    Fully qualified repos are stored as 'user/repo', and bot accounts are ignored.

    Parameters:
    -----------
    seed_user : str
        The starting user for exploration
    depth : int
        Maximum depth of exploration
    repo_limit : int
        Maximum number of repositories to fetch per user
    contrib_limit : int
        Maximum number of contributors to fetch per repository
    graph_file : str, optional
        Path to save the graph file
    state_file : str, optional
        Path to save/load the exploration state file for resuming
    """
    import pickle
    import os

    # Initialize or load state
    if state_file and os.path.exists(state_file):
        # Resume from saved state
        logger.info(f"Resuming exploration from state file: {state_file}")
        with open(state_file, "rb") as f:
            state = pickle.load(f)

        visited = state["visited"]
        exploration_queue = state["queue"]
        graph_file = state.get("graph_file", graph_file)

        # Load the graph
        if os.path.exists(graph_file):
            G = nx.read_gexf(graph_file)
            logger.info(f"Loaded graph with {len(G.nodes())} nodes and {len(G.edges())} edges from {graph_file}")
        else:
            logger.warning(f"Graph file {graph_file} not found. Starting with empty graph.")
            G = nx.Graph()
    else:
        # Start fresh
        G = nx.Graph()
        visited = set()
        exploration_queue = [(seed_user, 0)]  # (user, depth)

        logger.info(
            f"Starting graph building with seed user: {seed_user}",
            extra={"user": seed_user}
        )

    logger.info(
        f"Parameters - Depth: {depth}, Repo limit: {repo_limit}, Contributor limit: {contrib_limit}"
    )

    async with aiohttp.ClientSession() as session:
        # Process the exploration queue
        while exploration_queue:
            user, d = exploration_queue.pop(0)

            # Skip if already visited or beyond depth limit
            if user in visited or d > depth:
                continue

            # Skip bots immediately
            if "[bot]" in user.lower():
                logger.debug(f"Skipping bot user '{user}' entirely")
                continue

            visited.add(user)
            indent = "  " * d
            logger.info(f"{indent}Exploring user '{user}' at depth {d}",
                        extra={"depth": d, "user": user})

            # Fetch repositories for this user
            try:
                repos = await get_repos(session, user, repo_limit)
                logger.info(f"{indent}Found {len(repos)} repositories for user '{user}'",
                            extra={"depth": d, "user": user})
            except RateLimitExceeded as e:
                logger.error(f"{indent}Rate limit exceeded for user '{user}': {str(e)}")
                # Save state for resuming
                if state_file:
                    save_exploration_state(G, visited, [(user, d)] + exploration_queue, state_file)
                else:
                    save_graph(G, graph_file)
                return G
            except aiohttp.ClientError as e:
                logger.error(f"{indent}Failed to fetch repositories for user '{user}': {str(e)}")
                continue

            # Process each repository
            for repo in repos:
                full_repo = f"{user}/{repo}"  # unique repo ID

                logger.info(f"{indent}Processing repository: {full_repo}",
                            extra={"depth": d, "user": user, "repo": full_repo})

                try:
                    contributors = await get_contributors(session, user, repo, contrib_limit)
                    logger.info(f"{indent}Found {len(contributors)} contributors for {full_repo}",
                                extra={"depth": d, "user": user, "repo": full_repo})
                except RateLimitExceeded as e:
                    logger.error(f"{indent}Rate limit exceeded for repository '{full_repo}': {str(e)}")
                    # Save state for resuming
                    if state_file:
                        save_exploration_state(G, visited, [(user, d)] + exploration_queue, state_file)
                    else:
                        save_graph(G, graph_file)
                    return G
                except aiohttp.ClientError as e:
                    logger.warning(f"{indent}Failed to fetch contributors for {full_repo}: {str(e)}")
                    continue

                # Filter out bots right away
                human_contributors = [c for c in contributors if "[bot]" not in c.lower()]
                if len(human_contributors) < len(contributors):
                    logger.debug(f"{indent}Skipped {len(contributors) - len(human_contributors)} bots in {full_repo}")

                # Add contributors as nodes
                nodes_added = 0
                for c in human_contributors:
                    if c not in G:
                        G.add_node(c, repos=set())
                        nodes_added += 1
                    G.nodes[c]["repos"].add(full_repo)

                if nodes_added > 0:
                    logger.info(f"{indent}Added {nodes_added} new nodes for repository {full_repo}")

                # Connect collaborators (undirected)
                edges_added = 0
                for i, c1 in enumerate(human_contributors):
                    for c2 in human_contributors[i + 1:]:
                        G.add_edge(c1, c2, repo=full_repo)
                        edges_added += 1

                if edges_added > 0:
                    logger.info(f"{indent}Added {edges_added} edges for repository {full_repo}")

                # Add contributors to the exploration queue for the next depth level
                for contributor in human_contributors:
                    if contributor not in visited:
                        exploration_queue.append((contributor, d + 1))

            # Periodically save progress
            if len(visited) % 10 == 0:
                if state_file:
                    save_exploration_state(G, visited, exploration_queue, state_file)
                else:
                    save_graph(G, graph_file)

    logger.info(f"Graph building completed. Final graph has {len(G.nodes())} nodes and {len(G.edges())} edges.")

    # Save final state
    if state_file:
        save_exploration_state(G, visited, [], state_file)
    else:
        save_graph(G, graph_file)

    return G


In [4]:
# -----------------------------------------------------
# Configuration
# -----------------------------------------------------
load_dotenv()
BASE = "https://api.github.com"
#Fetch token from env file
TOKEN = os.getenv("TOKEN_OLIVER")
HEADERS = {"Authorization": f"token {TOKEN}"} if TOKEN else {}
GRAPH_FILE = "github_graph_4.gexf"
SEED_USER = "gh05tdog"
DEPTH = 2
REPO_LIMIT = 40
CONTRIBUTOR_LIMIT = 400

# Define an async function to run the graph building process
async def run_graph_building(state_file=None):
    """
    Run the graph building process with support for resuming from a saved state.

    Parameters:
    -----------
    state_file : str, optional
        Path to the state file for resuming exploration

    Returns:
    --------
    G : nx.Graph
        The built graph
    """
    # Default state file if not provided
    if state_file is None:
        state_file = "exploration_state.pkl"

    # Check if we have a state file to resume from
    if os.path.exists(state_file):
        logger.info(f"Found exploration state file: {state_file}")
        G = await build_graph(SEED_USER, depth=DEPTH, contrib_limit=CONTRIBUTOR_LIMIT, 
                             repo_limit=REPO_LIMIT, graph_file=GRAPH_FILE, state_file=state_file)
    # If no state file but graph file exists, just load the graph
    elif os.path.exists(GRAPH_FILE):
        logger.info(f"Loading existing graph from {GRAPH_FILE}")
        G = nx.read_gexf(GRAPH_FILE)
    # Start fresh
    else:
        logger.info("Building new graph...")
        G = await build_graph(SEED_USER, depth=DEPTH, contrib_limit=CONTRIBUTOR_LIMIT, 
                             repo_limit=REPO_LIMIT, graph_file=GRAPH_FILE, state_file=state_file)

        # If the graph was saved during building due to rate limit, it's already in the right format
        # Otherwise, save it now
        if not os.path.exists(GRAPH_FILE):
            G_save = save_graph(G, GRAPH_FILE)
            G = G_save

    return G


In [10]:
# Resume Functionality
"""
## Rate Limit Handling and Resume Functionality

This notebook now includes functionality to handle GitHub API rate limits by saving the exploration state
and allowing you to resume exactly where you left off. Here's how it works:

1. When a rate limit is hit, the current state (graph, visited users, and exploration queue) is saved to a file.
2. When you run the notebook again, it automatically detects the saved state and resumes exploration.
3. The exploration continues from exactly where it left off, maintaining the same depth and exploration queue.

### Key Files:
- `exploration_state.pkl`: Contains the exploration state (visited users and queue)
- `exploration_state.gexf`: Contains the graph at the time the rate limit was hit

### How to Resume:
- If a rate limit is hit, just run the notebook again - it will automatically resume.
- If you manually interrupt the process (Ctrl+C), you can also resume by running the notebook again.
- The resume functionality remembers:
  - Which users have been visited
  - Which users are still in the queue to be explored
  - The depth of each user in the queue
  - The current graph structure

This ensures that you can build large graphs even when hitting rate limits, without losing progress.
"""

# Define state file path
STATE_FILE = "exploration_state.pkl"

# Run the async function using asyncio
try:
    G = asyncio.run(run_graph_building(state_file=STATE_FILE))
except KeyboardInterrupt:
    logger.info("Process interrupted by user. Progress has been saved and can be resumed.")
    # Try to load the saved state or graph if it exists
    if os.path.exists(STATE_FILE):
        logger.info(f"You can resume exploration by running the notebook again - state file {STATE_FILE} will be used.")
        # Try to load the saved graph if it exists
        import pickle
        with open(STATE_FILE, "rb") as f:
            state = pickle.load(f)
        graph_file = state.get("graph_file", GRAPH_FILE)
        if os.path.exists(graph_file):
            G = nx.read_gexf(graph_file)
        else:
            logger.error("No saved graph found. Exiting.")
            raise
    elif os.path.exists(GRAPH_FILE):
        logger.info(f"Loading saved graph from {GRAPH_FILE}")
        G = nx.read_gexf(GRAPH_FILE)
    else:
        logger.error("No saved graph or state found. Exiting.")
        raise

logger.info(f"Graph statistics: {len(G.nodes())} nodes, {len(G.edges())} edges")


[2025-10-29 20:42:21,594] INFO: Building new graph...
[2025-10-29 20:42:21,595] [USER:gh05tdog] INFO: Starting graph building with seed user: gh05tdog
[2025-10-29 20:42:21,596] INFO: Parameters - Depth: 2, Repo limit: 40, Contributor limit: 400
[2025-10-29 20:42:21,597] [DEPTH:0][USER:gh05tdog] INFO: Exploring user 'gh05tdog' at depth 0
[2025-10-29 20:42:22,031] [DEPTH:0][USER:gh05tdog] INFO: Found 11 repositories for user 'gh05tdog'
[2025-10-29 20:42:22,032] [DEPTH:0][USER:gh05tdog][REPO:gh05tdog/DAPM_Master_Thesis_Group_D] INFO: Processing repository: gh05tdog/DAPM_Master_Thesis_Group_D
[2025-10-29 20:42:22,301] [DEPTH:0][USER:gh05tdog][REPO:gh05tdog/DAPM_Master_Thesis_Group_D] INFO: Found 11 contributors for gh05tdog/DAPM_Master_Thesis_Group_D
[2025-10-29 20:42:22,302] INFO: Added 11 new nodes for repository gh05tdog/DAPM_Master_Thesis_Group_D
[2025-10-29 20:42:22,303] INFO: Added 55 edges for repository gh05tdog/DAPM_Master_Thesis_Group_D
[2025-10-29 20:42:22,304] [DEPTH:0][USER:gh

In [11]:
# How many projects are a user involved in?
user_project_counts = {node: len(data.get("repos", "").split(", ")) if data
                          .get("repos") else 0 for node, data in G.nodes(data=True)}
sorted_users = sorted(user_project_counts.items(), key=lambda x: x[1], reverse=True)
logger.info("Top users by number of projects contributed to:")
for i, (user, count) in enumerate(sorted_users[:10]):
    logger.info(f"  {i+1}. {user}: {count} projects", extra={"user": user})

# If you want to see more users, uncomment and run this cell
# for i, (user, count) in enumerate(sorted_users[10:100]):
#     logger.info(f"  {i+11}. {user}: {count} projects", extra={"user": user})


[2025-10-29 21:19:40,730] INFO: Top users by number of projects contributed to:
[2025-10-29 21:19:40,731] [USER:cmorty] INFO:   1. cmorty: 177 projects
[2025-10-29 21:19:40,732] [USER:nfi] INFO:   2. nfi: 176 projects
[2025-10-29 21:19:40,732] [USER:jimparis] INFO:   3. jimparis: 168 projects
[2025-10-29 21:19:40,733] [USER:darconeous] INFO:   4. darconeous: 166 projects
[2025-10-29 21:19:40,733] [USER:g-oikonomou] INFO:   5. g-oikonomou: 164 projects
[2025-10-29 21:19:40,733] [USER:mmuman] INFO:   6. mmuman: 162 projects
[2025-10-29 21:19:40,734] [USER:pabigot] INFO:   7. pabigot: 161 projects
[2025-10-29 21:19:40,734] [USER:remyleone] INFO:   8. remyleone: 159 projects
[2025-10-29 21:19:40,735] [USER:malvira] INFO:   9. malvira: 156 projects
[2025-10-29 21:19:40,735] [USER:adamdunkels] INFO:   10. adamdunkels: 155 projects


In [3]:
import networkx as nx

def shortest_path_between_users(G, source_user, target_user):
    """
    Find the shortest path (in hops) from source_user to target_user.

    Parameters:
        G (nx.Graph): The GitHub collaboration graph.
        source_user (str): Starting username.
        target_user (str): Target username.

    Returns:
        path (list): List of users from source_user to target_user.
    """
    if source_user not in G:
        raise ValueError(f"User '{source_user}' not found in graph")

    if target_user not in G:
        raise ValueError(f"User '{target_user}' not found in graph")

    try:
        path = nx.shortest_path(G, source=source_user, target=target_user)
        return path
    except nx.NetworkXNoPath:
        raise nx.NetworkXNoPath(
            f"No path found from '{source_user}' to '{target_user}'"
        )


In [ ]:
def remove_bots(G, save=False):
    """
    Remove all bot accounts (nodes containing '[bot]') from the graph.
    Returns a cleaned copy.
    """
    bots = [n for n in G.nodes if "[bot]" in n.lower()]
    print(f"🤖 Removing {len(bots)} bot nodes...")
    G_clean = G.copy()
    G_clean.remove_nodes_from(bots)
    print(f"✅ Graph now has {len(G_clean.nodes())} nodes and {len(G_clean.edges())} edges.")
    if save:
        save_graph(G_clean, "cleaned_graph.gexf")
    return G_clean


In [ ]:
G_clean = remove_bots(G, save=True)
G = G_clean


In [4]:
# Example usage of the new function:
shortest_path_between_users(nx.read_gexf("cleaned_graph.gexf"), "gh05tdog", "torvalds")


NetworkXNoPath: No path found from 'gh05tdog' to 'torvalds'